In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Activation, Dropout, Flatten, Dense
from tensorflow.keras.layers import Conv2D, MaxPooling2D
import cv2
# Extracting the compressed dataset.
from zipfile import ZipFile 
data_path = '/content/drive/MyDrive/Colab Notebooks/archive.zip' 
with ZipFile(data_path, 'r') as zip:
  zip.extractall()
  # path to the folder containing our dataset

dataset = 'traffic_Data/DATA'

# path of label file
data = 'https://drive.google.com/file/d/1GRxegq7zdUGt_wFavzTPcrp3Oxop3HKK/view?usp=sharing'
labelfile = pd.read_csv(data, header=None, sep='\n')
labelfile = labelfile[0].str.split(',', expand=True)

In [3]:

train_ds = tf.keras.preprocessing.image_dataset_from_directory(dataset, validation_split=0.2, subset='training',
                                                              image_size=(224,224), seed=123, batch_size=32)
val_ds = tf.keras.preprocessing.image_dataset_from_directory(dataset, validation_split=0.2, subset='validation',
                                                              image_size=(224,224), seed=123, batch_size=32)

Found 4170 files belonging to 58 classes.
Using 3336 files for training.
Found 4170 files belonging to 58 classes.
Using 834 files for validation.


In [ ]:
class_number = train_ds.class_names
class_names = []
for i in class_number:
  class_names.append(labelfile[int(i)])

plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    for i in range(25):
        ax = plt.subplot(5, 5, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")

plt.show()

In [5]:
data_aug = tf.keras.Sequential(
    [
        tf.keras.layers.experimental.preprocessing.RandomFlip('horizontal', input_shape=(224,224,3)),
        tf.keras.layers.experimental.preprocessing.RandomRotation(0.1),
        tf.keras.layers.experimental.preprocessing.RandomZoom(0.2),
        tf.keras.layers.experimental.preprocessing.RandomFlip(mode='horizontal_and_vertical')
])

model = Sequential()
model.add(data_aug)
model.add(tf.keras.layers.Rescaling(1./255))
model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))
model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))
model.add(Conv2D(256, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))
model.add(Flatten())
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(128, activation='relu'))
model.add(Dense(len(labelfile), activation='softmax'))

from keras.utils import plot_model
tf.keras.utils.plot_model(
    model,
    show_shapes=True,
    show_dtype=True,
    show_layer_activations=True
)

model.compile(optimizer='adam', loss=tf.keras.losses.SparseCategoricalCrossentropy(), metrics=['accuracy'])

from keras.callbacks import ModelCheckpoint, EarlyStopping
mycallback = [EarlyStopping(monitor='val_loss', patience=5)]
epoch = 50
history = model.fit(train_ds, validation_data=val_ds, callbacks=mycallback, epochs=epoch)

Epoch 1/50
105/105 [==============================] - 35s 233ms/step - loss: 3.3513 - accuracy: 0.1838 - val_loss: 2.2461 - val_accuracy: 0.3861
Epoch 2/50
105/105 [==============================] - 23s 216ms/step - loss: 2.3165 - accuracy: 0.3258 - val_loss: 1.8233 - val_accuracy: 0.4341
Epoch 3/50
105/105 [==============================] - 22s 210ms/step - loss: 1.9865 - accuracy: 0.3822 - val_loss: 1.5688 - val_accuracy: 0.4916
Epoch 4/50
105/105 [==============================] - 22s 211ms/step - loss: 1.7358 - accuracy: 0.4535 - val_loss: 1.4524 - val_accuracy: 0.5336
Epoch 5/50
105/105 [==============================] - 23s 218ms/step - loss: 1.5899 - accuracy: 0.4943 - val_loss: 1.2736 - val_accuracy: 0.6007
Epoch 6/50
105/105 [==============================] - 23s 213ms/step - loss: 1.4629 - accuracy: 0.5090 - val_loss: 1.1479 - val_accuracy: 0.6475
Epoch 7/50
105/105 [==============================] - 23s 213ms/step - loss: 1.3929 - accuracy: 0.5459 - val_loss: 1.1324 - val_ac